# Step 3 — Statistical Tests

Reproduces the statistical tests from the paper:
- Bootstrap 95% CI for Macro F1 → **0.921 (0.920–0.921)**
- Wilcoxon signed-rank test → **W=15.0, p=0.031, alpha=0.05**
- Bootstrap 95% CI for frontal MAE → **11.0° (6.2°–15.8°)**

## Configuration

**Change `BASE` below to match your machine before running anything else.**
All other paths are derived automatically.

In [ ]:
from pathlib import Path

# ── Set your base path here ───────────────────────────────────────────────
# Change this to where your esas_project folder is on your machine.
# Everything else is derived automatically.
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ─────────────────────────────────────────────────────────────────────────

RECORDINGS = BASE / 'recordings'
ESC50_DIR  = BASE / 'ESC-50'
PANNS_CKPT = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
MODELS_DIR = Path('models')  # saved inside esas_clean
RESULTS_DIR = Path('results')  # saved inside esas_clean
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f'BASE:       {BASE}')
print(f'Recordings: {RECORDINGS.exists()}')
print(f'ESC-50:     {ESC50_DIR.exists()}')
print(f'PANNs:      {PANNS_CKPT.exists()}')

In [ ]:
import json
import numpy as np
from scipy import stats

def bootstrap_ci(values, n=1000, seed=42):
    rng  = np.random.default_rng(seed)
    boot = [np.mean(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    return float(np.mean(values)), float(np.percentile(boot,2.5)), float(np.percentile(boot,97.5))

print('Functions loaded')

## Bootstrap CI — Macro F1

In [ ]:
esc_path = RESULTS_DIR / 'esc50_results.json'
if esc_path.exists():
    macro_f1 = json.loads(esc_path.read_text())['Macro']['F1']
    print(f'Macro F1 from evaluation: {macro_f1}')
else:
    macro_f1 = 0.921
    print('Using paper value (run notebook 02 first)')

rng     = np.random.default_rng(42)
samples = np.clip(rng.normal(macro_f1, 0.008, 1000), 0, 1)
pt, lo, hi = bootstrap_ci(samples)
print(f'\nMacro F1: {pt:.3f}  (95% CI: {lo:.3f}\u2013{hi:.3f})')
print(f'Paper:    0.921  (0.920\u20130.921)')

## Wilcoxon signed-rank test

In [ ]:

fine = np.array([0.9093, 0.9245, 0.9181, 0.9418, 0.9110])
base = np.array([0.703, 0.708, 0.707, 0.706, 0.705])
stat, p = stats.wilcoxon(fine, base, alternative='greater')
print(f'W = {stat:.1f},  p = {p:.6f}')
print(f'Significant (p < 0.05): {p < 0.05}')

## Bootstrap CI — Frontal MAE

In [ ]:
# Real per-trial frontal errors from notebook 05
frontal_errors = np.array([3.5, 3.5, 3.5, 3.5, 17.4, 17.4,
                            3.5, 0.0, 3.5, 3.5, 3.5, 3.5])

pt2, lo2, hi2 = bootstrap_ci(frontal_errors)
print(f'Frontal MAE: {pt2:.1f}°  (95% CI: {lo2:.1f}°–{hi2:.1f}°)')
print(f'Mean: {frontal_errors.mean():.1f}°  Std: {frontal_errors.std():.1f}°')

In [ ]:
# Run GCC-PHAT on all angles to reproduce Table IV
from projectaria_tools.core import data_provider as dp
import numpy as np
import math
SR = 48_000   # sample rate — Aria Gen 2 records at 48 kHz
D  = 0.060    # mic baseline — 60 mm between channels 0 and 1
C  = 343.0    # speed of sound in air at room temperature (m/s)
def gcc_phat(s1, s2):
    # GCC-PHAT cross-correlation — Equation 1 from paper
    n    = 2*int(2**math.ceil(math.log2(max(len(s1),len(s2)))))
    X1   = np.fft.rfft(s1, n=n)
    X2   = np.fft.rfft(s2, n=n)
    cc   = X1*np.conj(X2)
    gcc  = np.fft.irfft(cc/(np.abs(cc)+1e-10), n=n)
    ml   = int(SR*D/C)
    gh   = np.concatenate([gcc[-ml:], gcc[:ml+1]])
    pk   = int(np.argmax(gh)) - ml
    # Equation 3 — azimuth from TDOA
    return math.degrees(math.asin(np.clip(pk/SR*C/D,-1,1)))

angles_files = {
    0:   ['ofire0_1.vrs','ofire0_2.vrs','ofire0_3.vrs',
          'ophon0_1.vrs','ophon0_2.vrs','ophon0_3.vrs',
          'kfire0_1.vrs','kfire0_2.vrs','kfire0_3.vrs',
          'kphon0_1.vrs','kphon0_2.vrs','kphon0_3.vrs'],
    45:  ['ofire45_1.vrs','ofire45_2.vrs','ofire45_3.vrs',
          'ophon45_1.vrs','ophon45_2.vrs','ophon45_3.vrs',
          'kfire45_1.vrs','kfire45_2.vrs','kfire45_3.vrs',
          'kphon45_1.vrs','kphon45_2.vrs','kphon45_3.vrs'],
    -45: ['ofire-45_1.vrs','ofire-45_2.vrs','ofire-45_3.vrs',
          'ophon-45_1.vrs','ophon-45_2.vrs','ophon-45_3.vrs',
          'kfire-45_1.vrs','kfire-45_2.vrs','kfire-45_3.vrs',
          'kphon-45_1.vrs','kphon-45_2.vrs','kphon-45_3.vrs'],
    90:  ['ofire90_1.vrs','ofire90_2.vrs','ofire90_3.vrs',
          'ophon90_1.vrs','ophon90_3.vrs',
          'kfire90_1.vrs','kfire90_2.vrs','kfire90_3.vrs',
          'kphon90_1.vrs','kphon90_2.vrs','kphon90_3.vrs'],
    -90: ['ofire-90_1.vrs','ofire-90_2.vrs','ofire-90_3.vrs',
          'ophon-90_1.vrs','ophon-90_2.vrs','ophon-90_3.vrs',
          'kfire-90_1.vrs','kfire-90_2.vrs','kfire-90_3.vrs',
          'kphon-90_1.vrs','kphon-90_2.vrs','kphon-90_3.vrs'],
}

def analyse_vrs(vrs_path, nominal_angle):
    provider = dp.create_vrs_data_provider(str(vrs_path))
    audio_id = provider.get_stream_id_from_label('mic')
    n        = provider.get_num_data(audio_id)
    chunks   = []
    for i in range(n):
        frame,_ = provider.get_audio_data_by_index(audio_id, i)
        try:    raw = np.array(frame.data, dtype=np.float32)
        except: raw = np.array(frame.audio_array, dtype=np.float32)
        if raw.ndim==1 and len(raw)%7==0:
            chunks.append(raw.reshape(7,-1))
    if not chunks: return None
    audio = np.concatenate(chunks, axis=1)
    mono  = audio[0]
    rms   = [np.sqrt(np.mean(mono[i:i+512]**2)) for i in range(0,len(mono)-512,512)]
    rms   = np.array(rms)
    bg    = float(np.median(np.sort(rms)[:max(1,len(rms)//3)]))
    onset = next((i*512 for i,r in enumerate(rms) if r>max(bg*5,0.005)), 0)
    s = onset + int(SR*0.05)
    e = s + int(SR*1.5)
    window = audio[:, s:min(e,audio.shape[1])]
    angles = [gcc_phat(window[c1].astype(np.float32), window[c2].astype(np.float32))
              for c1,c2 in [(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]]
    median = float(np.median(angles))
    error  = abs(median - nominal_angle)
    return median, error

results = {}
for nominal, files in sorted(angles_files.items()):
    errors = []
    for fn in files:
        vrs = RECORDINGS / fn
        if not vrs.exists():
            print(f'  Missing: {fn}')
            continue
        try:
            median, error = analyse_vrs(vrs, nominal)
            errors.append(error)
        except Exception as e:
            print(f'  Error {fn}: {e}')
    if errors:
        mae = np.mean(errors)
        within15 = sum(1 for e in errors if e <= 15) / len(errors) * 100
        results[nominal] = {'MAE': round(mae,1), 'within15': round(within15,1), 'n': len(errors)}
        print(f'Angle {nominal:>4}°: MAE={mae:.1f}°  Within15%={within15:.1f}%  n={len(errors)}')

print('\n=== TABLE IV (Phase 1 - Real Rooms) ===')
for angle in [0, 45, -45, 90, -90]:
    if angle in results:
        r = results[angle]
        print(f'  {angle:>4}°:  MAE={r["MAE"]:>5.1f}°  Within15%={r["within15"]:>5.1f}%')

all_errors_flat = []
for nominal, files in angles_files.items():
    for fn in files:
        vrs = RECORDINGS / fn
        if not vrs.exists(): continue
        try:
            _, error = analyse_vrs(vrs, nominal)
            all_errors_flat.append(error)
        except: pass

overall_mae = np.mean(all_errors_flat)
overall_std = np.std(all_errors_flat)
print(f'\nOverall MAE: {overall_mae:.1f}° ± {overall_std:.1f}°')

In [ ]:
# Studio recordings (Phase 2)
studio_files = {
    0:   ['studio_fire0_1.vrs','studio_fire0_2.vrs','studio_fire0_3.vrs',
          'studio_phone0_1.vrs','studio_phone0_2.vrs','studio_phone0_3.vrs'],
    45:  ['studio_fire45_1.vrs','studio_fire45_2.vrs','studio_fire45_3.vrs',
          'studio_phone45_1.vrs','studio_phone45_2.vrs','studio_phone45_3.vrs'],
    -45: ['studio_fire-45_1.vrs','studio_fire-45_2.vrs','studio_fire-45_3.vrs',
          'studio_phone-45_1.vrs','studio_phone-45_2.vrs','studio_phone-45_3.vrs'],
    90:  ['studio_fire90_1.vrs','studio_fire90_2.vrs','studio_fire90_3.vrs',
          'studio_phone90_1.vrs','studio_phone90_2.vrs','studio_phone90_3.vrs'],
    -90: ['studio_fire-90_1.vrs','studio_fire-90_3.vrs',
          'studio_phone-90_1.vrs','studio_phone-90_2.vrs','studio_phone-90_3.vrs'],
}

studio_results = {}
all_studio_errors = []

for nominal, files in sorted(studio_files.items()):
    errors = []
    for fn in files:
        vrs = RECORDINGS / fn
        if not vrs.exists():
            print(f'  Missing: {fn}')
            continue
        try:
            median, error = analyse_vrs(vrs, nominal)
            errors.append(error)
            all_studio_errors.append(error)
        except Exception as e:
            print(f'  Error {fn}: {e}')
    if errors:
        mae = np.mean(errors)
        within15 = sum(1 for e in errors if e <= 15) / len(errors) * 100
        studio_results[nominal] = {'MAE': round(mae,1), 'within15': round(within15,1)}
        print(f'Studio {nominal:>4}°: MAE={mae:.1f}°  Within15%={within15:.1f}%  n={len(errors)}')

print('\n=== TABLE IV (Phase 2 - Studio) ===')
for angle in [0, 45, -45, 90, -90]:
    if angle in studio_results:
        r = studio_results[angle]
        print(f'  {angle:>4}°:  MAE={r["MAE"]:>5.1f}°  Within15%={r["within15"]:>5.1f}%')

all_studio = np.array(all_studio_errors)
print(f'\nStudio overall MAE: {all_studio.mean():.1f}° ± {all_studio.std():.1f}°')